# Predicting Student Test Scores - Regression Baseline Models

## Objective
Build and compare baseline regression models to predict student test scores.

##  Models in this Notebook
1. **Simple Linear Regression** - Single best feature baseline
2. **Multiple Linear Regression** - All features baseline
3. **Ridge Regression** - L2 regularization
4. **Lasso Regression** - L1 regularization (feature selection)
5. **ElasticNet** - Combined L1 + L2
6. **Polynomial Regression** - Non-linear relationships


## Evaluation Matrics
- **RMSE (Root Mean Squared Error)** - Primary metric
- **MAE (Mean Absolute Error)** - Interpretable error
- **R2 Score** - Variance explained


## Previous Work
- [EDA Notebook](https://github.com/mahbub-04/PS-S6E1-Predicting-Student-Test-Scores/blob/main/notebooks/01_eda.ipynb) - Data exploration and insights

---

**Author:** Md Mahbub Ur Rashid

In [1]:
# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine learning - Regression models
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score

# Metrics
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Warning
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10,6)

In [2]:
# Load datasets
train = pd.read_csv('/kaggle/input/playground-series-s6e1/train.csv')
test = pd.read_csv('/kaggle/input/playground-series-s6e1/test.csv')

# Display basic info
print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")
print(f"\n{'='*50}")
print("First few rows:")
train.head()


Train shape: (630000, 13)
Test shape: (270000, 12)

First few rows:


,id,age,gender,course,study_hours,class_attendance,internet_access,sleep_hours,sleep_quality,study_method,facility_rating,exam_difficulty,exam_score
0,0,21,female,b.sc,7.91,98.8,no,4.9,average,online videos,low,easy,78.3
1,1,18,other,diploma,4.95,94.8,yes,4.7,poor,self-study,medium,moderate,46.7
2,2,20,female,b.sc,4.68,92.6,yes,5.8,poor,coaching,high,moderate,99.0
3,3,19,male,b.sc,2.00,49.5,yes,8.3,average,group study,high,moderate,63.9
4,4,23,male,bca,7.65,86.9,yes,9.6,good,self-study,high,easy,100.0


## Dataset Overview

From our EDA, we know:
- **Target variable:** `exam_score` (continuous, 19.6 - 100)
- **Features:** 11 features (4 numerical, 7 categorical)
- **No missing values** 
- **Strongest correlation:** `study_hours` (0.76)

### Feature Types:
- **Numerical:** age, study_hours, class_attendance, sleep_hours
- **Categorical:** gender, course, internet_access, sleep_quality, study_method, facility_rating, exam_difficulty


In [3]:
# Separate feature and target
X = train.drop(['id','exam_score'],axis=1)
y = train['exam_score']

X_test = test.drop(['id'],axis=1)

print(f'Features shape: {X.shape}')
print(f'Target shape: {y.shape}')
print(f'\nFeature columns:\n{X.columns.tolist()}')

Features shape: (630000, 11)
Target shape: (630000,)

Feature columns:
['age', 'gender', 'course', 'study_hours', 'class_attendance', 'internet_access', 'sleep_hours', 'sleep_quality', 'study_method', 'facility_rating', 'exam_difficulty']


##  Feature Engineering

### Step 1: Label Encoding (Ordinal Features)
Features with natural order will be label encoded:
- `sleep_quality`: poor (1) → average (2) → good (3)
- `facility_rating`: low (1) → medium (2) → high (3)
- `exam_difficulty`: easy (1) → moderate (2) → hard (3)


In [4]:
# Copy data to avoid modifying original
X_processed = X.copy()
X_test_processed = X_test.copy()

print("Original X shape:", X.shape)
print("Processed X shape:", X_processed.shape)


Original X shape: (630000, 11)
Processed X shape: (630000, 11)


In [5]:
# Define mapping for sleep_quality
sleep_mapping = {'poor': 1, 'average': 2, 'good': 3}

# Apply mapping
X_processed['sleep_quality'] = X_processed['sleep_quality'].map(sleep_mapping)
X_test_processed['sleep_quality'] = X_test_processed['sleep_quality'].map(sleep_mapping)

# Check results
print("Before encoding, values were:", X['sleep_quality'].unique())
print("After encoding, values are:", X_processed['sleep_quality'].unique())


Before encoding, values were: ['average' 'poor' 'good']
After encoding, values are: [2 1 3]


In [6]:
# facility_rating encoding
facility_mapping = {'low': 1, 'medium': 2, 'high': 3}
X_processed['facility_rating'] = X_processed['facility_rating'].map(facility_mapping)
X_test_processed['facility_rating'] = X_test_processed['facility_rating'].map(facility_mapping)

# exam_difficulty encoding
difficulty_mapping = {'easy': 1, 'moderate': 2, 'hard': 3}
X_processed['exam_difficulty'] = X_processed['exam_difficulty'].map(difficulty_mapping)
X_test_processed['exam_difficulty'] = X_test_processed['exam_difficulty'].map(difficulty_mapping)

# Verify all 3 ordinal features
print(" Label Encoding Complete!\n")
print("sleep_quality:", sorted(X_processed['sleep_quality'].unique()))
print("facility_rating:", sorted(X_processed['facility_rating'].unique()))
print("exam_difficulty:", sorted(X_processed['exam_difficulty'].unique()))


 Label Encoding Complete!

sleep_quality: [np.int64(1), np.int64(2), np.int64(3)]
facility_rating: [np.int64(1), np.int64(2), np.int64(3)]
exam_difficulty: [np.int64(1), np.int64(2), np.int64(3)]


### Step 2: One-Hot Encoding
Features without natural order will be one-hot encoded:
- `gender`: male, female, other
- `course`: b.sc, diploma, bca, etc.
- `internet_access`: yes, no
- `study_method`: online videos, self-study, coaching, etc.


In [7]:
# Define nominal (non-ordinal) features
nominal_features = ['gender', 'course', 'internet_access', 'study_method']

print("Features to one-hot encode:")
for col in nominal_features:
    print(f"  {col}: {X_processed[col].nunique()} categories")


Features to one-hot encode:
  gender: 3 categories
  course: 7 categories
  internet_access: 2 categories
  study_method: 5 categories


In [8]:
# Apply one-hot encoding using pd.get_dummies()
X_processed = pd.get_dummies(X_processed, columns=nominal_features, drop_first=True)
X_test_processed = pd.get_dummies(X_test_processed, columns=nominal_features, drop_first=True)

print(" One-Hot Encoding Complete!")
print(f"\nShape before encoding: (630000, 11)")
print(f"Shape after encoding: {X_processed.shape}")
print(f"\nNew columns created: {X_processed.shape[1] - 11}")


 One-Hot Encoding Complete!

Shape before encoding: (630000, 11)
Shape after encoding: (630000, 20)

New columns created: 9


In [9]:
# Check processed data


# Shape check :
print(f"X_processed: {X_processed.shape}")
print(f"X_test_processed: {X_test_processed.shape}")

# Columns check :
print(f"Columns : \n{X_processed.columns.tolist()[:10]}\n")

# Data types check :
print(f'Data types: \n{X_processed.dtypes.value_counts()}\n')

X_processed: (630000, 20)
X_test_processed: (270000, 20)
Columns : 
['age', 'study_hours', 'class_attendance', 'sleep_hours', 'sleep_quality', 'facility_rating', 'exam_difficulty', 'gender_male', 'gender_other', 'course_b.sc']

Data types: 
bool       13
int64       4
float64     3
Name: count, dtype: int64



In [10]:
X_processed.head()

,age,study_hours,class_attendance,sleep_hours,sleep_quality,facility_rating,exam_difficulty,gender_male,gender_other,course_b.sc,course_b.tech,course_ba,course_bba,course_bca,course_diploma,internet_access_yes,study_method_group study,study_method_mixed,study_method_online videos,study_method_self-study
0,21,7.91,98.8,4.9,2,1,1,False,False,True,False,False,False,False,False,False,False,False,True,False
1,18,4.95,94.8,4.7,1,2,2,False,True,False,False,False,False,False,True,True,False,False,False,True
2,20,4.68,92.6,5.8,1,3,2,False,False,True,False,False,False,False,False,True,False,False,False,False
3,19,2.00,49.5,8.3,2,3,2,True,False,True,False,False,False,False,False,True,True,False,False,False
4,23,7.65,86.9,9.6,3,3,1,True,False,False,False,False,False,True,False,True,False,False,False,True


##  Train-Test Split

We'll split data into:
- **Training set (80%):** To train models
- **Validation set (20%):** To evaluate models


In [11]:
# Split data into train and validation sets
X_train, X_val, y_train, y_val = train_test_split(
    X_processed,
    y,
    test_size = 0.2,
    random_state=42
)

print(f'Shape of training set : {X_train.shape}')
print(f'Shape of validation set : {X_val.shape}')

print(f"\nDistribution check:")
print(f"  Train - Mean: {y_train.mean():.2f}, Std: {y_train.std():.2f}")
print(f"  Val   - Mean: {y_val.mean():.2f}, Std: {y_val.std():.2f}")

Shape of training set : (504000, 20)
Shape of validation set : (126000, 20)

Distribution check:
  Train - Mean: 62.48, Std: 18.93
  Val   - Mean: 62.60, Std: 18.86


##  Model 1: Simple Linear Regression

**Goal:** Predict exam_score using only the strongest feature

**Feature:** `study_hours` (correlation = 0.76)

**Equation:** `exam_score = β₀ + β₁ × study_hours`

In [12]:
# Extract only study_hours for Simple Linear Regression
X_train_simple = X_train[['study_hours']]
X_val_simple = X_val[['study_hours']]

print(f'X_train_simple shape : {X_train_simple.shape}')
print(f'X_val_simple shape : {X_val_simple.shape}')

X_train_simple shape : (504000, 1)
X_val_simple shape : (126000, 1)


In [13]:
# Initialize and train Simple Linear Regression
simple_lr = LinearRegression()
simple_lr.fit(X_train_simple,y_train)

print('Simple Linear Regression trained!')

print(f'Model parameters :')
print(f' Intercept (β₀): {simple_lr.intercept_:.2f}')
print(f"  Coefficient (β₁): {simple_lr.coef_[0]:.2f}")
print(f"\nEquation: exam_score = {simple_lr.intercept_:.2f} + {simple_lr.coef_[0]:.2f} × study_hours")

Simple Linear Regression trained!
Model parameters :
 Intercept (β₀): 38.02
  Coefficient (β₁): 6.12

Equation: exam_score = 38.02 + 6.12 × study_hours


In [14]:
# Make predictions on validation set
y_pred_simple = simple_lr.predict(X_val_simple)

print("Predictions made!")
print(f"\nFirst 5 predictions:")
print(f"{'Actual':<20} {'Predicted':<20} {'Difference':<20}")
print("-" * 60)
for i in range(5):
    actual = y_val.iloc[i]
    predicted = y_pred_simple[i]
    diff = abs(actual - predicted)
    print(f"{actual:<20.2f} {predicted:<20.2f} {diff:<20.2f}")

Predictions made!

First 5 predictions:
Actual               Predicted            Difference          
------------------------------------------------------------
51.30                56.80                5.50                
50.60                58.08                7.48                
79.90                65.24                14.66               
55.40                63.65                8.25                
50.40                46.03                4.37                


In [15]:
# Calculate metrics
rmse = np.sqrt(mean_squared_error(y_val, y_pred_simple))
mae = mean_absolute_error(y_val, y_pred_simple)
r2 = r2_score(y_val, y_pred_simple)

print(f"\n Metrics (on validation set):")
print(f"  RMSE: {rmse:.2f}")
print(f"  MAE:  {mae:.2f}")
print(f"  R²:   {r2:.4f}")


 Metrics (on validation set):
  RMSE: 12.22
  MAE:  9.89
  R²:   0.5801


In [16]:
# Prepare test data (only study_hours)
X_test_simple = X_test_processed[['study_hours']]

# Make predictions on test set
test_pred_simple = simple_lr.predict(X_test_simple)

# Create submission dataframe
submission_simple = pd.DataFrame({
    'id': test['id'],
    'exam_score': test_pred_simple
})

# Save to CSV
submission_simple.to_csv('submission_simple_lr.csv', index=False)

print(" Simple LR Submission file created!")
print(f"\nFile: submission_simple_lr.csv")
print(f"Rows: {len(submission_simple)}")
print(f"\nPreview:")
print(submission_simple.head(10))


 Simple LR Submission file created!

File: submission_simple_lr.csv
Rows: 270000

Preview:
       id  exam_score
0  630000   79.920938
1  630001   78.452931
2  630002   78.391764
3  630003   56.555150
4  630004   50.438452
5  630005   76.067418
6  630006   59.980501
7  630007   57.227987
8  630008   77.718927
9  630009   85.242466


---

###  Simple Linear Regression - Results

**Model Details:**
- Feature: `study_hours` only
- Equation: `exam_score = 38.02 + 6.12 × study_hours`

**Performance Metrics:**

| Dataset | RMSE | MAE | R² |
|---------|------|-----|-----|
| Validation | 12.22 | 9.89 | 0.5801 |
| **Public Test** | **12.24** | - | - |

**Kaggle Leaderboard:**
- 🏆 **Rank:** 3216
- 📅 **Date:** January 25, 2026
- 📊 **Public Score:** 12.23983

**Key Insights:**
-  Validation and test scores very close (12.22 vs 12.24) - good generalization
-  58% variance explained by study_hours alone
-  Room for improvement - 42% variance still unexplained

**Next Step:** Multiple Linear Regression with all features

---


##  Model 2: Multiple Linear Regression

**Goal:** Predict exam_score using ALL features

**Features:** All 20 features (numerical + encoded categorical)

**Equation:** `exam_score = β₀ + β₁×feature₁ + β₂×feature₂ + ... + β₂₀×feature₂₀`

**Expected:** Better performance than Simple LR (using 1 feature vs 20 features)


In [17]:
# Train Multiple Linear Regression (using all features)
multiple_lr = LinearRegression()
multiple_lr.fit(X_train,y_train)

print(f'X_train shape : {X_train_simple.shape}')
print(f'X_val shape : {X_val_simple.shape}')

X_train shape : (504000, 1)
X_val shape : (126000, 1)


In [18]:
print(f"\nModel uses {X_train.shape[1]} features")
print(f"Intercept (β₀): {multiple_lr.intercept_:.2f}")


Model uses 20 features
Intercept (β₀): -2.33


In [19]:
# Get feature importance 
coefficients = pd.DataFrame({
    'Feature': X_train.columns,
    'Coefficient': multiple_lr.coef_
}).sort_values('Coefficient',key = abs, ascending =False)
print(coefficients.head())

                       Feature  Coefficient
19     study_method_self-study    -9.102641
18  study_method_online videos    -8.284140
16    study_method_group study    -7.248273
1                  study_hours     5.678064
4                sleep_quality     4.453087


In [20]:
# Make predictions on validation set
y_pred_multiple = multiple_lr.predict(X_val)
print("Predictions made!")

print(f"\nFirst 5 predictions comparison:")
print(f"{'Actual':<20} {'Predicted':<20} {'Difference':<20}")
print("-" * 60)
for i in range(5):
    actual = y_val.iloc[i]
    predicted = y_pred_multiple[i]
    diff = abs(actual - predicted)
    print(f"{actual:<20.2f} {predicted:<20.2f} {diff:<20.2f}")

Predictions made!

First 5 predictions comparison:
Actual               Predicted            Difference          
------------------------------------------------------------
51.30                60.57                9.27                
50.60                47.81                2.79                
79.90                61.93                17.97               
55.40                58.24                2.84                
50.40                48.30                2.10                


In [21]:
# Calculate metrics
rmse_multiple = np.sqrt(mean_squared_error(y_val, y_pred_multiple))
mae_multiple = mean_absolute_error(y_val, y_pred_multiple)
r2_multiple = r2_score(y_val, y_pred_multiple)

print("=" * 60)
print("MULTIPLE LINEAR REGRESSION PERFORMANCE")
print("=" * 60)

print(f"\n Validation Metrics:")
print(f"  RMSE: {rmse_multiple:.2f}")
print(f"  MAE:  {mae_multiple:.2f}")
print(f"  R²:   {r2_multiple:.4f}")

print(f"\n Comparison with Simple LR:")
print(f"  RMSE improvement: {12.22 - rmse_multiple:.2f} (↓ better)")
print(f"  R² improvement:   {r2_multiple - 0.5801:.4f} (↑ better)")

MULTIPLE LINEAR REGRESSION PERFORMANCE

 Validation Metrics:
  RMSE: 8.89
  MAE:  7.09
  R²:   0.7779

 Comparison with Simple LR:
  RMSE improvement: 3.33 (↓ better)
  R² improvement:   0.1978 (↑ better)


In [22]:
# Make predictions on test set (all features)
test_pred_multiple = multiple_lr.predict(X_test_processed)

# Create submission file
submission_multiple = pd.DataFrame({
    'id': test['id'],
    'exam_score': test_pred_multiple
})

# Save to CSV
submission_multiple.to_csv('submission_multiple_lr.csv', index=False)

print(" Multiple LR Submission file created!")
print(f"\nFile: submission_multiple_lr.csv")
print(f"Rows: {len(submission_multiple)}")
print(f"\nPreview:")
print(submission_multiple.head(10))


 Multiple LR Submission file created!

File: submission_multiple_lr.csv
Rows: 270000

Preview:
       id  exam_score
0  630000   71.849357
1  630001   69.573242
2  630002   87.492240
3  630003   54.743434
4  630004   47.178000
5  630005   71.267617
6  630006   73.037902
7  630007   57.981246
8  630008   79.087553
9  630009   92.423402


---

###  Multiple Linear Regression - Results

**Model Details:**
- Features: All 20 features
- Training samples: 504,000

**Performance Metrics:**

| Dataset | RMSE | MAE | R² |
|--------|------|-----|-----|
| Validation | 8.89 | 7.09 | 0.7779 |
| **Public Test** | **8.87318** | - | - |

**Improvement over Simple LR:**
- 📉 RMSE: 12.22 → 8.89 (↓ 3.33, 27% better)
- 📈 R²: 0.58 → 0.78 (+19.78% variance explained)

**Top 5 Important Features:**
1. study_hours: +5.68
2. sleep_quality: +4.45
3. study_method variations (negative coefficients - relative to baseline)

**Next Step:** Ridge Regression with regularization

---


## Ridge Regression

Ridge Regression adds L2 regularization to prevent overfitting.

**Key Points:**
- Penalty term: α × Σ(coefficients²)
- Starting with alpha=1.0


In [23]:
# Train Ridge Regression with L2 regularization
ridge = Ridge(alpha=1.0, random_state=42)
ridge.fit(X_train, y_train)

print('Ridge Regression trained!')

Ridge Regression trained!


In [24]:
print(f'X_train shape: {X_train.shape}')
print(f'X_val shape: {X_val.shape}')
print(f"\nModel uses {X_train.shape[1]} features")
print(f"Alpha (regularization): {ridge.alpha}")
print(f"Intercept (β₀): {ridge.intercept_:.2f}")

X_train shape: (504000, 20)
X_val shape: (126000, 20)

Model uses 20 features
Alpha (regularization): 1.0
Intercept (β₀): -2.33


In [25]:
# Get feature coefficients
ridge_coefficients = pd.DataFrame({
    'Feature': X_train.columns,
    'Coefficient': ridge.coef_
}).sort_values('Coefficient',key = abs,ascending = False)
print(ridge_coefficients.head(10))

                       Feature  Coefficient
19     study_method_self-study    -9.102278
18  study_method_online videos    -8.283779
16    study_method_group study    -7.247923
1                  study_hours     5.678064
4                sleep_quality     4.453075
17          study_method_mixed    -4.430716
5              facility_rating     3.635563
3                  sleep_hours     1.323685
2             class_attendance     0.312755
12                  course_bba     0.249436


In [26]:
# Make predictions on validation set
y_pred_ridge = ridge.predict(X_val)

print("Predictions made!")
print(f"\nFirst 5 predictions comparison:")
print(f"{'Actual':<20} {'Predicted':<20} {'Difference':<20}")
print("-" * 60)
for i in range(5):
    actual = y_val.iloc[i]
    predicted = y_pred_ridge[i]
    diff = abs(actual - predicted)
    print(f"{actual:<20.2f} {predicted:<20.2f} {diff:<20.2f}")


Predictions made!

First 5 predictions comparison:
Actual               Predicted            Difference          
------------------------------------------------------------
51.30                60.57                9.27                
50.60                47.81                2.79                
79.90                61.93                17.97               
55.40                58.24                2.84                
50.40                48.30                2.10                


In [27]:
# Calculate metrics
rmse_ridge = np.sqrt(mean_squared_error(y_val, y_pred_ridge))
mae_ridge = mean_absolute_error(y_val, y_pred_ridge)
r2_ridge = r2_score(y_val, y_pred_ridge)

print("=" * 60)
print("RIDGE REGRESSION PERFORMANCE")
print("=" * 60)

print(f"\n Validation Metrics:")
print(f"  RMSE: {rmse_ridge:.2f}")
print(f"  MAE:  {mae_ridge:.2f}")
print(f"  R²:   {r2_ridge:.4f}")


RIDGE REGRESSION PERFORMANCE

 Validation Metrics:
  RMSE: 8.89
  MAE:  7.09
  R²:   0.7779


In [28]:
print(f"\n Comparison with Multiple LR:")
print(f"  RMSE difference: {rmse_multiple - rmse_ridge:.2f}")
print(f"  R² difference:   {r2_ridge - r2_multiple:.4f}")


 Comparison with Multiple LR:
  RMSE difference: -0.00
  R² difference:   -0.0000


In [29]:
# Try different alpha values
alphas = [0.01, 0.1, 1, 10, 100, 1000]

print("Testing different alpha values:")
print("-" * 60)

for alpha in alphas:
    ridge_temp = Ridge(alpha=alpha, random_state=42)
    ridge_temp.fit(X_train, y_train)
    y_pred_temp = ridge_temp.predict(X_val)
    rmse_temp = np.sqrt(mean_squared_error(y_val, y_pred_temp))
    print(f"Alpha: {alpha:8} | RMSE: {rmse_temp:.4f}")


Testing different alpha values:
------------------------------------------------------------
Alpha:     0.01 | RMSE: 8.8870
Alpha:      0.1 | RMSE: 8.8870
Alpha:        1 | RMSE: 8.8870
Alpha:       10 | RMSE: 8.8870
Alpha:      100 | RMSE: 8.8870
Alpha:     1000 | RMSE: 8.8881


### Ridge Result
Tested alpha values from 0.01 to 1000. All gave same RMSE (8.887) as Multiple LR.
**Conclusion:** Dataset does not benefit from L2 regularization.
Skipping Ridge submission, moving to Lasso for feature selection approach.


## Lasso Regression

Lasso uses L1 regularization for automatic feature selection.

**Key Differences from Ridge:**
- Can set coefficients exactly to zero
- Performs feature selection
- Creates sparse models

**Goal:** Identify most important features and potentially improve generalization.


In [30]:
from sklearn.linear_model import Lasso

# Train Lasso Regression
lasso = Lasso(alpha=1.0, random_state=42)
lasso.fit(X_train, y_train)

print('Lasso Regression trained!')


Lasso Regression trained!


In [31]:
print(f'X_train shape: {X_train.shape}')
print(f'X_val shape: {X_val.shape}')
print(f"\nModel uses {X_train.shape[1]} features")
print(f"Alpha (regularization): {lasso.alpha}")
print(f"Intercept (β₀): {lasso.intercept_:.2f}")

X_train shape: (504000, 20)
X_val shape: (126000, 20)

Model uses 20 features
Alpha (regularization): 1.0
Intercept (β₀): -0.46


In [32]:
# Get feature coefficients
lasso_coefficients = pd.DataFrame({
    'Feature': X_train.columns,
    'Coefficient': lasso.coef_
}).sort_values('Coefficient', key=abs, ascending=False)

print("\nFeature coefficients (sorted by magnitude):")
print(lasso_coefficients)

# Count how many features set to zero
zero_features = (lasso.coef_ == 0).sum()
print(f"\nFeatures set to exactly zero: {zero_features} out of {len(lasso.coef_)}")



Feature coefficients (sorted by magnitude):
                       Feature  Coefficient
1                  study_hours     5.619305
4                sleep_quality     3.047031
5              facility_rating     2.155613
3                  sleep_hours     1.037514
2             class_attendance     0.316121
0                          age     0.000000
6              exam_difficulty     0.000000
7                  gender_male    -0.000000
8                 gender_other     0.000000
9                  course_b.sc    -0.000000
10               course_b.tech    -0.000000
11                   course_ba    -0.000000
12                  course_bba     0.000000
13                  course_bca     0.000000
14              course_diploma     0.000000
15         internet_access_yes    -0.000000
16    study_method_group study    -0.000000
17          study_method_mixed     0.000000
18  study_method_online videos    -0.000000
19     study_method_self-study    -0.000000

Features set to exactly zero: 

In [33]:
# Make predictions on validation set
y_pred_lasso = lasso.predict(X_val)

print("Predictions made with only 5 features!")
print(f"\nFirst 5 predictions comparison:")
print(f"{'Actual':<20} {'Predicted':<20} {'Difference':<20}")
print("-" * 60)
for i in range(5):
    actual = y_val.iloc[i]
    predicted = y_pred_lasso[i]
    diff = abs(actual - predicted)
    print(f"{actual:<20.2f} {predicted:<20.2f} {diff:<20.2f}")


Predictions made with only 5 features!

First 5 predictions comparison:
Actual               Predicted            Difference          
------------------------------------------------------------
51.30                57.21                5.91                
50.60                50.43                0.17                
79.90                60.54                19.36               
55.40                60.38                4.98                
50.40                49.76                0.64                


In [34]:
# Calculate metrics
rmse_lasso = np.sqrt(mean_squared_error(y_val, y_pred_lasso))
mae_lasso = mean_absolute_error(y_val, y_pred_lasso)
r2_lasso = r2_score(y_val, y_pred_lasso)

print("=" * 60)
print("LASSO REGRESSION PERFORMANCE")
print("=" * 60)

print(f"\n Validation Metrics:")
print(f"  RMSE: {rmse_lasso:.2f}")
print(f"  MAE:  {mae_lasso:.2f}")
print(f"  R²:   {r2_lasso:.4f}")

print(f"\n Model Simplification:")
print(f"  Features used: 5 out of 20")
print(f"  Features removed: 15")


LASSO REGRESSION PERFORMANCE

 Validation Metrics:
  RMSE: 9.68
  MAE:  7.77
  R²:   0.7367

 Model Simplification:
  Features used: 5 out of 20
  Features removed: 15


In [35]:
print(f"\n Comparison with Multiple LR:")
print(f"  Multiple LR RMSE: {rmse_multiple:.2f} (20 features)")
print(f"  Lasso RMSE:       {rmse_lasso:.2f} (5 features)")
print(f"  RMSE increase:    {rmse_lasso - rmse_multiple:.2f}")
print(f"")
print(f"  Multiple LR R²:   {r2_multiple:.4f}")
print(f"  Lasso R²:         {r2_lasso:.4f}")
print(f"  R² decrease:      {r2_lasso - r2_multiple:.4f}")



 Comparison with Multiple LR:
  Multiple LR RMSE: 8.89 (20 features)
  Lasso RMSE:       9.68 (5 features)
  RMSE increase:    0.79

  Multiple LR R²:   0.7779
  Lasso R²:         0.7367
  R² decrease:      -0.0412


In [36]:
# Try different alpha values
alphas = [0.001, 0.01, 0.1, 1, 10]

print("Testing different alpha values:")
print("-" * 80)
print(f"{'Alpha':<10} {'RMSE':<10} {'Features Used':<15}")
print("-" * 80)

for alpha in alphas:
    lasso_temp = Lasso(alpha=alpha, random_state=42)
    lasso_temp.fit(X_train, y_train)
    y_pred_temp = lasso_temp.predict(X_val)
    rmse_temp = np.sqrt(mean_squared_error(y_val, y_pred_temp))
    features_used = (lasso_temp.coef_ != 0).sum()
    print(f"{alpha:<10} {rmse_temp:<10.4f} {features_used}/20")


Testing different alpha values:
--------------------------------------------------------------------------------
Alpha      RMSE       Features Used  
--------------------------------------------------------------------------------
0.001      8.8870     19/20
0.01       8.8876     16/20
0.1        8.9453     9/20
1          9.6761     5/20
10         11.6804    2/20


In [37]:
# Train final Lasso with best alpha
lasso_final = Lasso(alpha=0.001, random_state=42)
lasso_final.fit(X_train, y_train)

print(f"Final Lasso Model (alpha=0.001)")
print(f"Features used: {(lasso_final.coef_ != 0).sum()}/20")

# Show which feature was removed
lasso_final_coef = pd.DataFrame({
    'Feature': X_train.columns,
    'Coefficient': lasso_final.coef_
})
zero_features = lasso_final_coef[lasso_final_coef['Coefficient'] == 0]
print(f"\nFeatures set to zero:")
print(zero_features)


Final Lasso Model (alpha=0.001)
Features used: 19/20

Features set to zero:
                Feature  Coefficient
15  internet_access_yes         -0.0


In [38]:
# Make predictions on test set
test_pred_lasso = lasso_final.predict(X_test_processed)

# Create submission file
submission_lasso = pd.DataFrame({
    'id': test['id'],
    'exam_score': test_pred_lasso
})

# Save to CSV
submission_lasso.to_csv('submission_lasso.csv', index=False)

print(" Lasso Submission file created!")
print(f"\nFile: submission_lasso.csv")
print(f"Alpha: 0.001")
print(f"Features used: 19/20")
print(f"Rows: {len(submission_lasso)}")
print(f"\nPreview:")
print(submission_lasso.head(10))

 Lasso Submission file created!

File: submission_lasso.csv
Alpha: 0.001
Features used: 19/20
Rows: 270000

Preview:
       id  exam_score
0  630000   71.865647
1  630001   69.545830
2  630002   87.492282
3  630003   54.740366
4  630004   47.154032
5  630005   71.280346
6  630006   73.048839
7  630007   57.990903
8  630008   79.085526
9  630009   92.403698


## Polynomial Regression

Polynomial features capture non-linear relationships and feature interactions.

**Approach:**
- Using degree=2 (quadratic terms + interactions)
- Creates squared terms and cross-products
- May capture diminishing returns in study hours, sleep hours

**Note:** Feature count will increase significantly.


In [39]:
from sklearn.preprocessing import PolynomialFeatures

# Create polynomial features (degree=2)
poly = PolynomialFeatures(degree=2, include_bias=False)
X_train_poly = poly.fit_transform(X_train)
X_val_poly = poly.transform(X_val)

print(f'Original features: {X_train.shape[1]}')
print(f'Polynomial features: {X_train_poly.shape[1]}')


Original features: 20
Polynomial features: 230


In [40]:
# Train with Ridge regularization (necessary for 230 features!)
poly_model = Ridge(alpha=1.0, random_state=42)
poly_model.fit(X_train_poly, y_train)

print('Polynomial model trained!')
print(f'Total parameters: {len(poly_model.coef_)}')


Polynomial model trained!
Total parameters: 230


In [41]:
# Make predictions on validation set
y_pred_poly = poly_model.predict(X_val_poly)

print("Predictions made!")
print(f"\nFirst 5 predictions comparison:")
print(f"{'Actual':<20} {'Predicted':<20} {'Difference':<20}")
print("-" * 60)
for i in range(5):
    actual = y_val.iloc[i]
    predicted = y_pred_poly[i]
    diff = abs(actual - predicted)
    print(f"{actual:<20.2f} {predicted:<20.2f} {diff:<20.2f}")


Predictions made!

First 5 predictions comparison:
Actual               Predicted            Difference          
------------------------------------------------------------
51.30                60.65                9.35                
50.60                47.65                2.95                
79.90                61.88                18.02               
55.40                59.04                3.64                
50.40                48.96                1.44                


In [42]:
# Calculate metrics
rmse_poly = np.sqrt(mean_squared_error(y_val, y_pred_poly))
mae_poly = mean_absolute_error(y_val, y_pred_poly)
r2_poly = r2_score(y_val, y_pred_poly)

print("=" * 60)
print("POLYNOMIAL REGRESSION PERFORMANCE")
print("=" * 60)

print(f"\n Validation Metrics:")
print(f"  RMSE: {rmse_poly:.2f}")
print(f"  MAE:  {mae_poly:.2f}")
print(f"  R²:   {r2_poly:.4f}")

print(f"\n Model Complexity:")
print(f"  Features: 230 (20 original + polynomial terms)")
print(f"  Regularization: Ridge (alpha=1.0)")


POLYNOMIAL REGRESSION PERFORMANCE

 Validation Metrics:
  RMSE: 8.88
  MAE:  7.09
  R²:   0.7784

 Model Complexity:
  Features: 230 (20 original + polynomial terms)
  Regularization: Ridge (alpha=1.0)


In [43]:
print(f"\n Comparison with Multiple LR:")
print(f"  Multiple LR RMSE: {rmse_multiple:.2f} (20 features)")
print(f"  Polynomial RMSE:  {rmse_poly:.2f} (230 features)")
print(f"  RMSE improvement: {rmse_multiple - rmse_poly:.4f}")
print(f"")
print(f"  Multiple LR R²:   {r2_multiple:.4f}")
print(f"  Polynomial R²:    {r2_poly:.4f}")
print(f"  R² improvement:   {r2_poly - r2_multiple:.4f}")



 Comparison with Multiple LR:
  Multiple LR RMSE: 8.89 (20 features)
  Polynomial RMSE:  8.88 (230 features)
  RMSE improvement: 0.0089

  Multiple LR R²:   0.7779
  Polynomial R²:    0.7784
  R² improvement:   0.0004


In [44]:
# Transform test data to polynomial
X_test_poly = poly.transform(X_test_processed)

# Make predictions on test set
test_pred_poly = poly_model.predict(X_test_poly)

# Create submission file
submission_poly = pd.DataFrame({
    'id': test['id'],
    'exam_score': test_pred_poly
})

# Save to CSV
submission_poly.to_csv('submission_polynomial.csv', index=False)

print(" Polynomial Submission file created!")
print(f"\nFile: submission_polynomial.csv")
print(f"Features: 230 (degree=2 polynomial)")
print(f"Rows: {len(submission_poly)}")
print(f"\nPreview:")
print(submission_poly.head(10))


 Polynomial Submission file created!

File: submission_polynomial.csv
Features: 230 (degree=2 polynomial)
Rows: 270000

Preview:
       id  exam_score
0  630000   71.858955
1  630001   69.688107
2  630002   86.543666
3  630003   55.063699
4  630004   47.640128
5  630005   72.139624
6  630006   72.983993
7  630007   58.579377
8  630008   79.073624
9  630009   91.952718
